In [ ]:
import os
from pathlib import Path
import urllib.request
import tarfile

import torch
from models.yolo3 import Darknet_53
from torch.nn import CrossEntropyLoss
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.notebook import tqdm

ROOT = Path().cwd().parent.joinpath("data")
ROOT

WindowsPath('h:/Programming/Machine_learning/deep machine learning/VIDEO_CURSE/data')

In [3]:
import urllib.request
import tarfile

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [5]:
url = "https://s3.amazonaws.com/fast-ai-imageclas/imagewoof2-320.tgz"
urllib.request.urlretrieve(url, "imagewoof2-320.tgz")

with tarfile.open("imagewoof2-320.tgz") as f:
    f.extractall(ROOT)

In [ ]:
model = Darknet_53(classes=20)
if torch.cuda.device_count() > 1:
    print(f"Используется {torch.cuda.device_count()} GPU")

model = model.to(device)
loss_model = CrossEntropyLoss()
optim = AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scaler = torch.amp.GradScaler("cuda")

EPOCHS = 100
scheduler = ReduceLROnPlateau(optim, mode="min", factor=0.5, patience=10)

c:\Users\1\miniconda3\envs\ml_main\Lib\site-packages\torch\amp\grad_scaler.py:132: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(


In [ ]:
import torchvision
from torchvision.transforms import v2
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, ConcatDataset, Dataset

class LabelOffsetDataset(Dataset):
    def __init__(self, dataset, offset):
        self.dataset = dataset
        self.offset = offset

    def __len__(self):
        return len(self.dataset)

    def __getitem(self, idx):
        img, label = self.dataset[idx]
        return img, label + self.offset

IMG_SIZE = 416
BATCH_SIZE = 64

train_transform = v2.Compose([
    v2.Resize((IMG_SIZE, IMG_SIZE)),
    v2.RandomHorizontalFlip(),
    v2.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = v2.Compose([
    v2.Resize((IMG_SIZE, IMG_SIZE)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_set_im = ImageFolder(ROOT.joinpath("imagenette2-320/train"), transform=train_transform)
val_set_im = ImageFolder(ROOT.joinpath("imagenette2-320/val"), transform=val_transform)

train_set_if = ImageFolder(ROOT.joinpath("imagewoof2-320/train"), transform=train_transform)
val_set_if = ImageFolder(ROOT.joinpath("imagewoof2-320/val"), transform=val_transform)

train_set_if = LabelOffsetDataset(train_set_if, offset=10)
val_set_if = LabelOffsetDataset(val_set_if, offset=10)

train_set = ConcatDataset([train_set_im, train_set_if])
val_set = ConcatDataset([val_set_im, val_set_if])

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

c:\Users\1\miniconda3\envs\ml_main\Lib\site-packages\torchvision\transforms\v2\_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


In [ ]:
train_loss_history, val_loss_history = list(), list()
train_acc_history, val_acc_history = list(), list()

loop = tqdm(range(EPOCHS), leave=True, desc="Training Darknet_53")
for epoch in loop:
    loss_epoch = []
    total = 0
    correct = 0 

    model.train()
    loop_train = tqdm(enumerate(train_loader), leave=False, desc=f"Train [{epoch+1}/{EPOCHS}]", total=len(train))